# Myllia - echoes of silenced genes
---

**authors**: [fsb2210](https://www.kaggle.com/fsb2210), [julianc93](https://www.kaggle.com/julianc93)

## Introduction

In [1]:
import os
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.impute import KNNImputer
from tqdm import tqdm

Define global options:

In [2]:
# directory with data files
data_dir = "../data"
working_dir = "../data"

# random state integer value for reproducibility concerns
random_state = 42

# neighbors to use for imputation of NaN expressions
n_neighbors = 3

# threshold for considering a gene to have been perturbed
threshold = 0.2

# weights for core training data vs external data
core_weight = 0.7
ext_weight = 0.3

## Data loading

In [3]:
# train & validation sets
raw_train_df = pd.read_csv(f"{data_dir}/training_data_means.csv")
valid_df = pd.read_csv(f"{data_dir}/pert_ids_val.csv")

In [4]:
# K562 dataset
raw_k562_df = pd.read_csv(f"{data_dir}/k562_data_means.csv")
# Hepg2 dataset
raw_hepg2_df = pd.read_csv(f"{data_dir}/hepg2_data_means.csv")
# Jurkat dataset
raw_jurkat_df = pd.read_csv(f"{data_dir}/jurkat_data_means.csv")
# RPE1 dataset
raw_rpe1_df = pd.read_csv(f"{data_dir}/rpe1_data_means.csv")

In [5]:
# load pre-fetched gene embeddings using ESM-2
gene_embeddings_df = pd.read_csv(f"{data_dir}/esm2_gene_embeddings.csv")

### Preprocessing steps

Grab the name of the output genes:

In [6]:
output_genes = raw_train_df.columns[1:]

Impute missing values in K562 and HepG2 datasets, using nearest beighbors in the training sample:

In [7]:
def impute(y, y_true, n_neighbors):
    # define imputer
    imputer = KNNImputer(n_neighbors=n_neighbors, weights="uniform")
    # stack y (with NaNs) and y_true
    y_stacked = np.vstack([y_true, y])
    y_imputed = imputer.fit_transform(y_stacked)
    y_ = y_imputed.copy()
    return y_[y_true.shape[0]:]

In [8]:
start_time = time.time()

if len(os.listdir("/home/fsbn/.cache/myllia/")) == 5:
    Y_train = np.load("/home/fsbn/.cache/myllia/y_train.npy")
    Y_k562 = np.load("/home/fsbn/.cache/myllia/y_k562.npy")
    Y_hepg2 = np.load("/home/fsbn/.cache/myllia/y_hepg2.npy")
    Y_jurkat = np.load("/home/fsbn/.cache/myllia/y_jurkat.npy")
    Y_rpe1 = np.load("/home/fsbn/.cache/myllia/y_rpe1.npy")
else:
    Y_train = raw_train_df.iloc[:, 1:].values
    Y_k562 = impute(raw_k562_df.iloc[:, 1:].values, Y_train, n_neighbors)
    Y_hepg2 = impute(raw_hepg2_df.iloc[:, 1:].values, Y_train, n_neighbors)
    Y_jurkat = impute(raw_jurkat_df.iloc[:, 1:].values, Y_train, n_neighbors)
    Y_rpe1 = impute(raw_rpe1_df.iloc[:, 1:].values, Y_train, n_neighbors)
    np.save("/home/fsbn/.cache/myllia/y_train.npy", Y_train)
    np.save("/home/fsbn/.cache/myllia/y_k562.npy", Y_k562)
    np.save("/home/fsbn/.cache/myllia/y_hepg2.npy", Y_hepg2)
    np.save("/home/fsbn/.cache/myllia/y_jurkat.npy", Y_jurkat)
    np.save("/home/fsbn/.cache/myllia/y_rpe1.npy", Y_rpe1)

training_time = time.time() - start_time
print(f"Data imputation completed in {training_time:.2f} seconds ({training_time/60:.2f} minutes)")

Data imputation completed in 0.08 seconds (0.00 minutes)


Replace NaNs with newly computed values from imputation

In [9]:
raw_k562_df[output_genes] = Y_k562
raw_hepg2_df[output_genes] = Y_hepg2
raw_jurkat_df[output_genes] = Y_jurkat
raw_rpe1_df[output_genes] = Y_rpe1

Grab the name of the perturbations in each of the datasets

In [10]:
k562_pert_genes = raw_k562_df.pert_symbol.tolist()[:-1]
hepg2_pert_genes = raw_hepg2_df.pert_symbol.tolist()[:-1]
jurkat_pert_genes = raw_jurkat_df.pert_symbol.tolist()[:-1]
rpe1_pert_genes = raw_rpe1_df.pert_symbol.tolist()[:-1]

Compute delta expressions with respect to the `non-targeting` row for every dataset:

In [11]:
# train dataset
train_base = raw_train_df.iloc[-1,1:].values
train_df = raw_train_df[output_genes].apply(lambda x: x - train_base, axis=1)
train_df.insert(0, "pert_symbol", raw_train_df["pert_symbol"].copy())
# train_df.drop([train_df.shape[0]-1], inplace=True)
train_df.loc[train_df.shape[0]-1, output_genes] = train_base
train_df["source"] = "myllia"

# K562 dataset
k562_base = raw_k562_df.iloc[-1,1:].values
k562_df = raw_k562_df[output_genes].apply(lambda x: x - k562_base, axis=1)
k562_df.insert(0, "pert_symbol", raw_k562_df["pert_symbol"].copy())
# k562_df.drop([k562_df.shape[0]-1], inplace=True)
k562_df.loc[k562_df.shape[0]-1, output_genes] = k562_base
k562_df["source"] = "k562"

# HepG2 dataset
hepg2_base = raw_hepg2_df.iloc[-1,1:].values
hepg2_df = raw_hepg2_df[output_genes].apply(lambda x: x - hepg2_base, axis=1)
hepg2_df.insert(0, "pert_symbol", raw_hepg2_df["pert_symbol"].copy())
# hepg2_df.drop([hepg2_df.shape[0]-1], inplace=True)
hepg2_df.loc[hepg2_df.shape[0]-1, output_genes] = hepg2_base
hepg2_df["source"] = "hepg2"

# Jurkat dataset
jurkat_base = raw_jurkat_df.iloc[-1,1:].values
jurkat_df = raw_jurkat_df[output_genes].apply(lambda x: x - jurkat_base, axis=1)
jurkat_df.insert(0, "pert_symbol", raw_jurkat_df["pert_symbol"].copy())
# jurkat_df.drop([jurkat_df.shape[0]-1], inplace=True)
jurkat_df.loc[jurkat_df.shape[0]-1, output_genes] = jurkat_base
jurkat_df["source"] = "jurkat"

# RPE1 dataset
rpe1_base = raw_rpe1_df.iloc[-1,1:].values
rpe1_df = raw_rpe1_df[output_genes].apply(lambda x: x - rpe1_base, axis=1)
rpe1_df.insert(0, "pert_symbol", raw_rpe1_df["pert_symbol"].copy())
# rpe1_df.drop([rpe1_df.shape[0]-1], inplace=True)
rpe1_df.loc[rpe1_df.shape[0]-1, output_genes] = rpe1_base
rpe1_df["source"] = "rpe1"

train_df.shape, k562_df.shape, hepg2_df.shape, jurkat_df.shape, rpe1_df.shape

((81, 5129), (2058, 5129), (2394, 5129), (2394, 5129), (2394, 5129))

## Feature and target engineering

We do some preprocessing on the input data to make it useful.

In particular, we will compute the delta expressions per-gene (DEG) frequencies (per-gene per-dataset) and combine them using a weigthed average (with more weight assigned to the training dataset, `core_weight`). Thus, the result of such operation is an array of shape (5127,), that is, the length of `output_genes`.

In addition, we compute mean expressions per sample, again per-dataset.

In [12]:
# map datasets
data_dict = {
    "train": train_df,
    "k562": k562_df,
    "hepg2": hepg2_df,
    "jurkat": jurkat_df,
    "rpe1": rpe1_df,
}

In [13]:
gene_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()

# DEG frequency of each gene per dataset
deg_freq_per_dataset = {}
for source_name, current_df in data_dict.items():
    deg_freq_per_dataset[source_name] = (np.abs(current_df[gene_cols]) > threshold).mean(axis=0)

# combine with weighting
deg_freq_core = deg_freq_per_dataset["train"]
deg_freq_ext = pd.concat([deg_freq_per_dataset[key] for key in data_dict.keys() if key != "train"], axis=1).mean(axis=1)
# weighted average
deg_freq_final = core_weight * deg_freq_core + ext_weight * deg_freq_ext

## mean baseline expression (control rows)
## control_rows = []
## for source_name, current_df in data_dict.items():
##     control_row = current_df[gene_cols].iloc[[-1]]
##     control_rows.append(control_row)
## concat control rows and compute mean per gene
## control_df = pd.concat(control_rows, axis=0)
## mean_baseline_final = control_df.mean(axis=0)

# single-row DataFrames for broadcasting
freq_df = pd.DataFrame([deg_freq_final.values], columns=[f"freq_{c}" for c in gene_cols])
## mean_df = pd.DataFrame([mean_baseline_final.values], columns=[f"mean_{c}" for c in gene_cols])

In [14]:
# DEG frequency on the training dataset only
# gene_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
# training_deg_freq = (np.abs(train_df[gene_cols]) > threshold).mean(axis=0)
# freq_df = pd.DataFrame([training_deg_freq.values], columns=[f"freq_{c}" for c in gene_cols])

final_chunks, delta_chunks = [], []
for source_name, current_df in data_dict.items():
    common_genes = current_df["pert_symbol"].tolist()
    available_embeddings = gene_embeddings_df[gene_embeddings_df["pert_symbol"].isin(common_genes)]

    # merge deltas + embeddings
    merged_df = pd.merge(current_df, available_embeddings, on="pert_symbol", how="inner")

    # DEG frequency across perturbations, shape of (1, 5127)
    # freq_deg = (np.abs(current_df[gene_cols]) > threshold).mean(axis=0)

    # mean expression per sample
    # merged_df["mean_expr"] = merged_df[gene_cols].mean(axis=1).values
    # global magnitude of each specific perturbation
    perturbation_mean_values = merged_df[gene_cols].mean(axis=1).values
    perturbation_mean_df = pd.DataFrame(perturbation_mean_values, columns=["mean_expr"])

    # broadcast DEG frequency from training set
    n_rows = len(merged_df)
    # freq_repeated = pd.concat([freq_df] * n_rows, ignore_index=True)
    freq_broadcast = pd.concat([freq_df] * n_rows, ignore_index=True)
    ## mean_broadcast = pd.concat([mean_df] * n_rows, ignore_index=True)

    # reset indices to ensure clean concatenation
    # merged_df = merged_df.reset_index(drop=True)
    # freq_repeated = freq_repeated.reset_index(drop=True)
    merged_df = merged_df.reset_index(drop=True)
    ## freq_broadcast = freq_broadcast.reset_index(drop=True)
    ## mean_broadcast = mean_broadcast.reset_index(drop=True)
    ## perturbation_mean_df = perturbation_mean_df.reset_index(drop=True)

    # concat: embeddings + mean(1) + freq(5127)
    # merged_df = pd.concat([merged_df, freq_repeated], axis=1)
    ## merged_df = pd.concat([merged_df, freq_broadcast, mean_broadcast], axis=1)
    merged_df = pd.concat([merged_df, freq_broadcast, perturbation_mean_df], axis=1)
    
    # save delta expressions separately
    delta_chunks.append(merged_df[gene_cols])
    # and drop them from the "X" dataset
    merged_df = merged_df.drop(columns=gene_cols)

    final_chunks.append(merged_df)

train_final = pd.concat(final_chunks, ignore_index=True)
train_final.drop(columns=["pert_symbol", "source"], inplace=True)
delta_train = pd.concat(delta_chunks, ignore_index=True)

Same as before but for the validation set

In [16]:
valid_df = valid_df.rename({"pert": "pert_symbol"}, axis=1)
valid_merged =  pd.merge(valid_df, gene_embeddings_df, on="pert_symbol", how="left")

# use mean values from train dataset
# valid_merged["mean_expr"] = train_df[gene_cols].mean(axis=1).mean()

valid_df = valid_df.rename({"pert": "pert_symbol"}, axis=1)
valid_merged = pd.merge(valid_df, gene_embeddings_df, on="pert_symbol", how="left")

# global mean
all_deltas = []
for source_name, current_df in data_dict.items():
    common_genes = current_df["pert_symbol"].tolist()
    available_embeddings = gene_embeddings_df[gene_embeddings_df["pert_symbol"].isin(common_genes)]
    # merge deltas + embeddings
    merged_df = pd.merge(current_df, available_embeddings, on="pert_symbol", how="inner")
    means = merged_df[gene_cols].mean(axis=1).values
    all_deltas.extend(means)
global_mean_magnitude = np.mean(all_deltas)

# DEG frequency
n_valid_rows = len(valid_merged)
# freq_repeated = pd.concat([freq_df] * n_valid_rows, ignore_index=True)
freq_broadcast = pd.concat([freq_df] * n_valid_rows, ignore_index=True)
## mean_broadcast = pd.concat([mean_df] * n_valid_rows, ignore_index=True)
valid_mean_df = pd.DataFrame({"mean_expr": [global_mean_magnitude] * n_valid_rows})

# valid_merged = valid_merged.reset_index(drop=True)
# freq_repeated = freq_repeated.reset_index(drop=True)
valid_merged = valid_merged.reset_index(drop=True)
freq_broadcast = freq_broadcast.reset_index(drop=True)
## mean_broadcast = mean_broadcast.reset_index(drop=True)
valid_mean_df = valid_mean_df.reset_index(drop=True)

# concatenate
# valid_final = pd.concat([valid_merged, freq_repeated], axis=1)
## valid_final = pd.concat([valid_merged, freq_broadcast, mean_broadcast], axis=1)
valid_final = pd.concat([valid_merged, freq_broadcast, valid_mean_df], axis=1)

valid_final.drop(columns=["pert_symbol", "class", "pert_id"], inplace=True)

### Training and validation features

Finally, we can create our input and output (`X`, `Y`) datasets to be used for training a machine learning model.

- `X` will be the result of a diffusive perturbation for each of the training perturbations,
- `Y` is the delta expressions of the 5127 genes of the training set.

Same idea for the `X_val` and `Y_val` for the validation set.

In [17]:
X = train_final.values
X_val = valid_final.values
Y = delta_train.values

print(f"- training feature shape: {X.shape}, training output shape: {Y.shape}")
print(f"- validation feature shape: {X_val.shape}")

- training feature shape: (9297, 6408), training output shape: (9297, 5127)
- validation feature shape: (60, 6408)


And save them into files:

In [18]:
np.save("../data/processed/X_esm2_stats_train.npy", X)
np.save("../data/processed/X_esm2_stats_val.npy", X_val)
np.save("../data/processed/Y_esm2_stats_train.npy", Y)